In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import json
import openpyxl
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
bath_path = './check.xlsx'
feature_path = './chosen_features_label.txt' ## Modify file to match the equivalent column names in the batch file
model_folder = './'
with open(feature_path, 'r') as file:
    chosen_features_label = [line[:-1] if line.endswith('\n') else line for line in file] 

with open('./abbs.json', 'r') as f:
    column_abb = json.load(f)
print('Number of features: ',len(chosen_features_label)-2) 

Number of features:  13


In [3]:
df0 = pd.read_excel(bath_path, engine='openpyxl')
df0.rename(columns={
    "Does the patient have choledocholithiasis (stone OR sludge in the common bile duct)?    A few notes:     1. Mark yes if they have a common bile duct stone on ERCP, MRCP, CT scan, or intra-operative cholangiogram.   2. Also mark yes if there is concern that the patient had choledocholithiasis but that there is a past stone      examples include but are not limited to:            a. pancreatitis + transamanitis - concern for gallstone (aka biliary) pancreatitis without finding of stone on imaging/ERCP           b. gallbladder stones (cholelithiasis) + transaminitis + RUQ pain without stone on imaging/ERCP (likely passed stone)  3. Also mark yes if they have cholangitis secondary to choledocholithiasis.":
        'Label'
}, inplace=True)
df = df0[chosen_features_label].copy()
df.rename(columns=column_abb, inplace=True)
df.replace({"Checked": "Yes", "Unchecked": "No"}, inplace=True)

## Drop unknown gender
valid_gender = df['Sex'].isin(['Male', 'Female'])
invalid_indices = df.index[~valid_gender]
print("Subjects with no gender values specified:", invalid_indices.tolist())
df = df.loc[valid_gender].copy()

df['Sex'] = df['Sex'].replace({'Female': 0, 'Male': 1}).astype('int8')
df['Sex'] = pd.to_numeric(df['Sex'], downcast='integer', errors='coerce')
df.replace({'Yes': 1, 'No': 0}, inplace=True)
df

Subjects with no gender values specified: []


/var/folders/n9/nrf9s6h16xs3pbl0jmsw7lnc0000gq/T/ipykernel_23718/85803237.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Sex'] = df['Sex'].replace({'Female': 0, 'Male': 1}).astype('int8')
/var/folders/n9/nrf9s6h16xs3pbl0jmsw7lnc0000gq/T/ipykernel_23718/85803237.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({'Yes': 1, 'No': 0}, inplace=True)


,Record ID,Choledocholithiasis,Sex,Cholangitis,Pancreatitis,Cholecystitis,AST,ALT,ALP,T. Bilirubin,Abd US,CBD stone on Abd US,CBD stone on CT scan,Charlson Comorbidity Index,Age
0,1,1,0,0.0,0.0,0.0,93,62,193,1.2,1,0,0,4.0,70
1,3,0,0,1.0,0.0,0.0,55,15,117,0.6,1,0,0,NaN,55
2,4,0,1,NaN,NaN,NaN,721,548,272,3.3,1,0,0,NaN,66
3,5,1,0,0.0,0.0,1.0,41,71,79,0.8,1,0,0,0.0,90
4,6,0,0,0.0,0.0,0.0,100,167,288,2.5,0,0,0,NaN,45
5,7,1,0,1.0,0.0,0.0,59,79,197,1.2,1,0,1,5.0,80


In [4]:
model = joblib.load(os.path.join(model_folder, 'initial.pkl'))
imputer = joblib.load(os.path.join(model_folder, 'iterative_imputer.pkl'))

X = df.drop(['Record ID','Choledocholithiasis'], axis=1)
X_imputed= imputer.transform(X)
X_imputed = pd.DataFrame(X_imputed, columns=X.columns)

In [5]:
def next_step(prob, cholangitis_flag):
    if cholangitis_flag == 1:
        return 'ERCP'
    if prob < 10:
        return 'CCY ± IOC'
    elif prob < 44:
        return 'MRCP'
    elif prob < 90:
        return 'EUS'
    else:
        return 'ERCP'

In [6]:
pred_probs = model.predict_proba(X_imputed) * 100  # convert to %
pred_probs = pred_probs.round(2)

# Probability of CBD (the positive class, column index 1)
prob_values = pred_probs[:, 1]

# Assign back to dataframe
df['Predicted Probability of CBD (%)'] = prob_values
df['Next Step Recommendation'] = [
    next_step(p, c) for p, c in zip(prob_values, X_imputed['Cholangitis'])
]
df.to_csv('./predictions.csv', index=False)
df

,Record ID,Choledocholithiasis,Sex,Cholangitis,Pancreatitis,Cholecystitis,AST,ALT,ALP,T. Bilirubin,Abd US,CBD stone on Abd US,CBD stone on CT scan,Charlson Comorbidity Index,Age,Predicted Probability of CBD (%),Next Step Recommendation
0,1,1,0,0.0,0.0,0.0,93,62,193,1.2,1,0,0,4.0,70,31.62,MRCP
1,3,0,0,1.0,0.0,0.0,55,15,117,0.6,1,0,0,NaN,55,35.42,ERCP
2,4,0,1,NaN,NaN,NaN,721,548,272,3.3,1,0,0,NaN,66,70.21,EUS
3,5,1,0,0.0,0.0,1.0,41,71,79,0.8,1,0,0,0.0,90,38.30,MRCP
4,6,0,0,0.0,0.0,0.0,100,167,288,2.5,0,0,0,NaN,45,54.69,EUS
5,7,1,0,1.0,0.0,0.0,59,79,197,1.2,1,0,1,5.0,80,96.73,ERCP
